# Agents

Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

An LLM Agent runs tools in a loop to achieve a goal. An agent runs until a stop condition is met - i.e., when the model emits a final output or an iteration limit is reached.

![Agent](./images/agent.png)

# Tools 
Tools give agents the ability to take actions. Agents go beyond simple model-only tool binding by facilitating:
 - Multiple tool calls in sequence (triggered by a single prompt)
 - Parallel tool calls when appropriate
 - Dynamic tool selection based on previous results
 - Tool retry logic and error handling
 - State persistence across tool calls

## Defining tools
Pass a list of tools to the agent.

In [1]:
from langchain.tools import tool
from langchain.agents import create_agent

### Search Tool 
https://docs.langchain.com/oss/python/integrations/tools


In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Rikkeisoft?")

'31 thg 12, 2025 · Rikkeisoft – A leading Vietnamese IT company providing software development, digital transformation, and AI solutions worldwide. XÁCH VALI VÀ ĐI DALI “ĐẬP ... [HN] ONE HR 15 thg 5, 2025 · Chặng đường thành công. Ngày 6 tháng 4 năm 2012, công ty Rikkeisoft được thành lập, đánh dấu “nơi giấc mơ bắt đầu” của bốn chàng trai trẻ. 7 thg 1, 2026 · Rikkei AI là công ty con của Rikkeisoft, được thành lập năm 2019 với tầm nhìn trở thành một trong những đơn vị dẫn đầu trong lĩnh vực AI, đồng hành cùng khách ... 10 thg 10, 2025 · Rikkeisoft đặt mục tiêu IPO tại Nhật Bản trong 3 năm tới, hướng đến giấc mơ trở thành “kỳ lân công nghệ Việt Nam”. Chiến lược Go Global, đầu tư AI và hợp ... 16 thg 7, 2025 · Công ty Rikkeisoft tuyển dụng vị trí Junior Automation Tester, trực tiếp tham gia vào các dự án hợp tác với đối tác Nhật Bản. Vị trí này yêu cầu ứng viên có khả ...'

In [6]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults(output_format="list")

search.invoke("Rikkeisoft?")

[{'snippet': 'We have been working with Rikkeisoft for 7 years, mainly in web and mobile application development. Rikkeisoft team can communicate fluently in Japanese and English, so we are assured of mutual understanding. In the future, we still want to keep a long-term relationship with Rikkeisoft .',
  'title': 'RIKKEI (THAILAND) CO., LTD - Rikkeisoft - Trusted IT Outsourcing Provider',
  'link': 'https://rikkeisoft.com/th/th/'},
 {'snippet': '231+ reviews môi trường làm việc, văn hoá, mức lương tại RIKKEISOFT . Được đăng ẩn danh bởi nhân viên làm việc tại đây',
  'title': '231+ Reviews RIKKEISOFT: Công ty có tốt không?',
  'link': 'https://1900.com.vn/danh-gia-dn/cong-ty-co-phan-rikkeisoft-1657'},
 {'snippet': 'Ông Tạ Sơn Tùng, Chủ tịch Rikkeisoft chia sẻ, Rikkeisoft đang chuẩn bị cho cột mốc IPO tại Nhật trong 3 năm tới và hướng tới giấc mơ trở thành kỳ lân công nghệ Việt Nam.',
  'title': 'Rikkeisoft tuyên bố IPO tại Nhật, hướng tới trở thành kỳ lân công nghệ ...',
  'link': 'htt

In [28]:
@tool
def search_rikkeisoft_information(query: str) -> str:
    """Search for information about Rikkeisoft."""
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    result = search.invoke(query)
    if result:
        return result
    else:
        return "No information found"

search_rikkeisoft_information.invoke({
    "query": "Rikkeisoft?"
})





'31 thg 12, 2025 · Rikkeisoft – A leading Vietnamese IT company providing software development, digital transformation, and AI solutions worldwide. [HN] ONE HR RIKKEI JAPAN LONG TRỌNG ... 15 thg 5, 2025 · Chặng đường thành công. Ngày 6 tháng 4 năm 2012, công ty Rikkeisoft được thành lập, đánh dấu “nơi giấc mơ bắt đầu” của bốn chàng trai trẻ. 16 thg 7, 2025 · Công ty Rikkeisoft tuyển dụng vị trí Junior Automation Tester, trực tiếp tham gia vào các dự án hợp tác với đối tác Nhật Bản. Vị trí này yêu cầu ứng viên có khả ... 10 thg 10, 2025 · Rikkeisoft đặt mục tiêu IPO tại Nhật Bản trong 3 năm tới, hướng đến giấc mơ trở thành “kỳ lân công nghệ Việt Nam”. 7 thg 1, 2026 · Rikkei AI là công ty con của Rikkeisoft, được thành lập năm 2019 với tầm nhìn trở thành một trong những đơn vị dẫn đầu trong lĩnh vực AI, đồng hành cùng khách ...'

### Get current time tool

In [29]:
@tool
def get_current_time(_: str="") -> str:
    """Get the current time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
get_current_time.invoke({})




'2026-02-01 12:05:27'

In [30]:

@tool
def calculate_years_of_establishment(start_year: int) -> str:
    """Calculate the number of years since the establishment of Rikkeisoft."""
    from datetime import datetime
    return f"Rikkeisoft was established in {datetime.now().year - start_year} years ago"

calculate_years_of_establishment.invoke({
    "start_year": 2010
})

'Rikkeisoft was established in 16 years ago'

## Model
The model is the reasoning engine of your agent. It can be specified in multiple ways, supporting both static and dynamic model selection.

### Static model
Static models are configured once when creating the agent and remain unchanged throughout execution. This is the most common and straightforward approach.

In [31]:
from config import settings
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model=settings.LLM_CHAT_MODEL,api_key=settings.LLM_API_KEY)

agent = create_agent(model=model, tools=[
    search_rikkeisoft_information,
    get_current_time,
    calculate_years_of_establishment
])


In [49]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time?"}]}
)

In [37]:
result

{'messages': [HumanMessage(content='what is the current time?', additional_kwargs={}, response_metadata={}, id='86b8c588-9188-407f-98f3-d8371a141305'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'6fd12077-8bbd-401b-befd-48ff1faa7987': 'EjQKMgFyyNp8tGoW0aAOd/sHC0Zp5LiBRYaooa9judGPPdWUnE8NQ9UxaNEV+Gk01r8EpnJR'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c179a-e574-7a62-bea3-f6344e7e5dfb-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': '6fd12077-8bbd-401b-befd-48ff1faa7987', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 134, 'output_tokens': 12, 'total_tokens': 146, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='2026-02-01 12:09:14', name='get_current_time', id='2000f0d1-5698-4304-bc44-de00230f8a54'

In [48]:
final_message = result["messages"][-1].content
print(final_message[0]["text"])

The current time is 12:09 PM on February 1, 2026.


In [50]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]}
)
result
final_message = result["messages"][-1].content
print(final_message[0]["text"])


Rikkeisoft được thành lập vào năm 2012. Tính đến hiện tại (năm 2026), công ty đã hoạt động được **14 năm**.


In [51]:
result

{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='e1615bd5-e67a-46b9-95ae-66a18e275205'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th\\u00e0nh l\\u1eadp Rikkeisoft"}'}, '__gemini_function_call_thought_signatures__': {'1bad3d91-e02d-487c-970f-a0844ff5bc7a': 'ErUCCrICAXLI2nyliDY4CV+8vKJgQLzMhFN42Msg8Ay8o3CSkeYvxvjWsGyqm5eSxnbHwCuwY8h1S3o6WnEc5NEOqr1QPJHcl4kyWVeBV0E/UG4B5TEUuKJ3FLiTcMoRR+M2rD4sGvlnppEzkNteN6Ssfc5yQ70dX+aKnOCS68BRx1xvcwqp2OOyXq2KsZct2lQQZyCvb4hAl9ZsEFEIaC7r1vHK3m6T9QDbeN3mQwY79XVe9zrfmWN+Ij8ZHNyvG7/0hsmLl7LxlseDHRnuP4OV8IR0gLcdiHUOBQU7R8Fe/XA1Jc7fciQAb6QE9fJiYT6MZ3jbOjtz7MXvOMST4HiV2Dyme7snkvXyR2R15M7P32/2Bv06pdN0HuUeBoC1zbRv9sZnjSuOr4y31SH2mpIiSo7oD1BJ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id=

### Dynamic Model là gì?

**Dynamic model** là cơ chế cho phép **chọn mô hình LLM tại thời điểm runtime** dựa trên **ngữ cảnh, trạng thái hiện tại hoặc logic tùy biến** (ví dụ: độ phức tạp câu hỏi, chi phí, độ trễ, người dùng trả phí hay miễn phí).  
Thay vì cố định một model (như GPT-4 hoặc Gemini-Pro), hệ thống có thể **tự động chuyển đổi model** để đạt hiệu quả tốt nhất giữa **chất lượng – chi phí – tốc độ**.

---

### Vì sao cần Dynamic Model?

Dynamic model giúp:
- **Routing thông minh**: câu hỏi đơn giản dùng model rẻ/nhanh, câu hỏi phức tạp dùng model mạnh.
-  **Tối ưu chi phí**: giảm dùng model đắt khi không cần thiết.
-  **Cải thiện hiệu năng**: ưu tiên model phản hồi nhanh trong các tình huống realtime.
- **Linh hoạt mở rộng**: dễ thêm model mới mà không thay đổi toàn bộ hệ thống.

### Middleware với `@wrap_model_call`


Để dùng dynamic model, bạn tạo middleware bằng decorator `@wrap_model_call`. Middleware này sẽ **chỉnh sửa model trong request** trước khi LLM được gọi.


In [52]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-2.5-flash")
advanced_model = ChatGoogleGenerativeAI(api_key=settings.LLM_API_KEY,model="gemini-2.5-pro")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection]
)

### System Prompt 
**System prompt** là thông điệp dùng để **định hình hành vi, vai trò và phong cách làm việc của agent** ngay từ đầu.  
Nó trả lời cho câu hỏi: *“Agent này nên suy nghĩ và phản hồi như thế nào?”*
Trong LangChain, có thể truyền system prompt khi tạo agent để kiểm soát:
- Cách agent tiếp cận nhiệm vụ
- Mức độ chi tiết / ngắn gọn
- Tính cách, vai trò (assistant, chuyên gia, reviewer, v.v.)

In [53]:
agent = create_agent(
    model=model,  # Default model
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    middleware=[dynamic_model_selection],
    system_prompt="You are a helpful assistant that can answer questions and help with tasks."
)

In [54]:
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current time?"}]}
)

{'messages': [HumanMessage(content='what is the current time?', additional_kwargs={}, response_metadata={}, id='47ea6cc2-bc05-47a9-9ca5-1fd202caf6e1'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'b4693717-0a2a-4724-99d4-a3471f8e3ec6': 'CskBAXLI2nwyxMw7v2CK/iIGSgyKlXko9ak5aquA1RL33j5+XDkH9ieR1+naat/rkVhKQ6IjiJkYJnZ1UOAMxczvINlnLjguwy4rgodTSmcgLAma0PoJ8p896mHZwUHiWdgkgfvITD+HgUiWP33PnDfR7uPSY6ubUiX/qfA7Au7AFOTzDxnYpZLrTrBcxU9O8xMZCl37g9YZXrm/HOejRhmZLr5qxomhdlaIKZkS8vwuOD1tUWnbO+77whwe9dwcUPx35SSNIhRlH0P7'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c1821-5bd5-7860-a3ea-73e127535895-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'b4693717-0a2a-4724-99d4-a3471f8e3ec6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'ou

# Structure Ouput Agent

- ToolStrategy: Sử dụng cho những model không hỗ trợ structure ouput, agent lấy kết quả và tự chuyện sang output mong muốn
- ProviderStrategy: Sử dụng với model hỗ trợ structure output, đáng tin cậy hơn.

In [62]:
from pydantic import BaseModel
from langchain.agents.structured_output import ProviderStrategy
class AgentOutput(BaseModel):
    answer: str
    time_of_answer: str

In [63]:
agent = create_agent(
    model=model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    response_format=ProviderStrategy(AgentOutput),
    system_prompt="You are a helpful assistant that can answer questions and help with tasks. You also return the time in the format of YYYY-MM-DD HH:MM:SS that you get from the current time tool."
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}]})
result


{'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='0a517193-8bd5-44e0-8f72-29b6a7120bba'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "n\\u0103m th\\u00e0nh l\\u1eadp Rikkeisoft"}'}, '__gemini_function_call_thought_signatures__': {'99337e57-27fb-451f-8597-03fbfbff9440': 'Er4ECrsEAXLI2nzNezCCZ9sT0sIt3iYKWLSrAeD8p7C3IU65MFYxLxSkPZed4ujP44fwFP96eNsgS5p/9PIEGTD/3RZ0kjZDjKqIbF70OzgKb5cL7Cm3N0iOoY3m/ZJec0cdsShrNopAdNPDrtsXUU9sLRwmp7EauG+sJTjznMFbcKwRvC15ne79AWbAFQcyIP2dAl38KirHTkaVvHSqdL0rak95qhEP+JsLm/XEIrEyi7bgZHw0HW69FCktgwPoaNFYRa+NPa+HtEZeIBGFW2XCbU+2Sp9mTMeXorSCic/+R0OyOHVoOOO7tNSRqS7XIiw6qELWtlxCA9TOjpp1iLZb6LTCV0dzyty9CrRmv6D6HiA7Gv2sh3ZFylSNV0c7RrUqskWek+RDJnvxB1OqVT9RQ1Rxgq38ux7b87qwc2Y6BSgypcUIP5XLeJlBmP63r2qU+oeHkUMvdz8coSDSRRW/C189i1xJ/MJhWV7zV+SWfJgV++i9Pq3swXILHmguu2Eenqag5heFU5d+t0GkbgygzkLWZX+OU1tyOmZmOiGChng65/S5K

In [64]:
print(result["structured_response"])

answer='Rikkeisoft được thành lập vào ngày 6 tháng 4 năm 2012. Tính đến nay (năm 2026), công ty đã hoạt động được 14 năm.' time_of_answer='2026-02-01 14:47:47'


### Memory

Trong LangChain, **Agent tự động duy trì lịch sử hội thoại** thông qua *message state*.  
Phần thông tin này có thể xem như **short-term memory** (bộ nhớ ngắn hạn) của agent, giúp agent:
- Hiểu ngữ cảnh cuộc trò chuyện
- Trả lời nhất quán qua nhiều lượt
- Tham chiếu lại thông tin đã nói trước đó


In [79]:
from langgraph.checkpoint.memory import MemorySaver  
checkpointer = MemorySaver()  # In-memory 
from typing import Any
agent = create_agent(
    model,
    tools=[search_rikkeisoft_information,get_current_time,calculate_years_of_establishment],
    checkpointer=checkpointer
)
config = {"configurable": {"thread_id": "session_1"}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Rikkeisoft được thành lập bao nhiêu năm rồi?"}],
}, config)

In [80]:
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Rikkeisoft được thành lập vào ngày **6 tháng 4 năm 2012**. Tính đến năm 2026, công ty đã hoạt động được **14 năm**.', 'extras': {'signature': 'EsQBCsEBAXLI2nzUIESNaXXr1UPFEak2Z3vJJHc+4fN4r1ZyrsaVld/KSPinoCEvRgp5KxpQ+xSw1OsGeodm30HFbMyoPpSqvi0hzaH+hElD5sUJJWGTGoy9DIQcQhxaaClWPZ1D1vsTMx3nVhpXNteludEbtKom/UsmwvzyNVyoXoys5UZayflkFd66G2igqN8uEL5jOCAN0s4cQ0n8gP6H4OykPPiZIdBkwgYh3JX03UazWhbbUJai8g8hMDbYZ13HgdjiFA=='}}]


In [81]:
#print checkpointer
checkpointer.get(config)

{'v': 4,
 'ts': '2026-02-01T08:19:52.418708+00:00',
 'id': '1f0ff46c-871e-67d1-8007-555f84fc1759',
 'channel_versions': {'__start__': '00000000000000000000000000000002.0.23085589289041608',
  'messages': '00000000000000000000000000000009.0.5606339938157182',
  'branch:to:model': '00000000000000000000000000000009.0.5606339938157182',
  '__pregel_tasks': '00000000000000000000000000000008.0.041211864309014246'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.6207690385828719'},
  'model': {'branch:to:model': '00000000000000000000000000000008.0.041211864309014246'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='58567d8b-3ced-4912-bfb9-8c32f20584f7'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "ng\\u00e

In [82]:
result_2 = agent.invoke({
    "messages": [{"role": "user", "content": "Tôi đã hỏi bạn gì nhỉ?"}],
}, config)
print(result_2["messages"][-1].content)
checkpointer.get(config)


[{'type': 'text', 'text': 'Bạn vừa hỏi tôi là: **"Rikkeisoft được thành lập bao nhiêu năm rồi?"**\n\nTôi đã trả lời rằng Rikkeisoft được thành lập vào ngày 6 tháng 4 năm 2012, và tính đến năm 2026 thì công ty đã hoạt động được 14 năm.', 'extras': {'signature': 'EtQFCtEFAXLI2nz6kuziwpUCucFPWAbZx/rqD9P9TDzBe88ylmiGrMpoulX35m/G2m4BHYGtTBsEH5Ge/yd38hSxT5tbMAZ9mXeh7iNpa0AQeF3dVPJmnGts36wSL9mg9hdQoqOssUjR5GXSbElONwWUidPa62uDdigbT4hQuF7FYn2mhDqKqrC1nMg6Xh/8kWW3bGLQ8Eyj0myqguRF2/PCu5F8YPfJk+IXHgJU4yWsC44W1MmXNTdc6NWfvfh6N3XiAvzmim11/kjGyp37eJx9L+wbPNVo4fEdtDri0FNLeWAgfOOwQmEnjsPBFoT87Wn4b485aHdu2+pHpylCuTptTwQKnc0C5lnG1LcffN9K6rcbUa3qMCf/hQv3/ZFLhNz6nQdSd+J2DgGrRq0eEwi5glJnY48GLp3QP5yCgfQZ+a/kQ6cxBzThJJmZSbX+CteHtlyh9SNtCb0F0XjLrFd5Y2ptlLJ+HziO9OzNVc/lljGpWNFWWtkSmfmF3hjHjGb8z6/J/Sb5bb4dhn0ofS5LUChQpvcWACMIpYcjH7Ij/LV1wScoYbE2Iq279rZ4wWDu3KZePyxXG7kdOGknuB7eRYeW0HLJ3kopZlKJlrSlpZ2AwumFW56jlK2xbZC2N1Aje6RGYEcvHuqqm2k+S+DeK0jXIx7/8oGBpyQPCfpmi1o4vXBbJaL+CAnosm2aBGB8VJrgRSjg5JZYxOAI3GOAPK5QitNILW

{'v': 4,
 'ts': '2026-02-01T08:19:54.821630+00:00',
 'id': '1f0ff46c-9e08-6fef-800a-3ca9c71ad1f5',
 'channel_versions': {'__start__': '00000000000000000000000000000011.0.9352225093053799',
  'messages': '00000000000000000000000000000012.0.6866417298599156',
  'branch:to:model': '00000000000000000000000000000012.0.6866417298599156',
  '__pregel_tasks': '00000000000000000000000000000008.0.041211864309014246'},
 'versions_seen': {'__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000010.0.8448645757642537'},
  'model': {'branch:to:model': '00000000000000000000000000000011.0.9352225093053799'},
  'tools': {}},
 'updated_channels': ['messages'],
 'channel_values': {'messages': [HumanMessage(content='Rikkeisoft được thành lập bao nhiêu năm rồi?', additional_kwargs={}, response_metadata={}, id='58567d8b-3ced-4912-bfb9-8c32f20584f7'),
   AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_rikkeisoft_information', 'arguments': '{"query": "ng\\u00e0y 

In [85]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Bây giờ là mấy giờ?"}]
}, config, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Bây giờ là mấy giờ?
Calling tools: ['get_current_time']
Agent: 2026-02-01 15:21:31
Agent: [{'type': 'text', 'text': 'Bây giờ là **15:21** (3 giờ 21 phút chiều), ngày **01 tháng 02 năm 2026**.', 'extras': {'signature': 'EjQKMgFyyNp83S5z15W4c9glNDoFc7nWCnfqvNZR+aeYU05ULeu/5DPm8ULpa8mr+N0Xki1W'}}]
